# Experiment — Traces
AA-01-03-E08 · Fictional disease-classification report · AI with review

## Introduction

A trace is a record of what happened during an interaction or run. It can include model requests and responses and, in larger agent systems, tool calls and other events. Inspecting it helps locate missing input or a claim unsupported by the supplied data. A trace does not expose hidden reasoning or prove the internal cause of an error ([OpenAI tracing documentation](https://openai.github.io/openai-agents-python/tracing/)).

In this experiment, you will inspect a report about a fictional disease-classification example. Four supplied outcome counts let you calculate accuracy, recall and missed positives without training a model. You will call `get_completion()`, keep the request and response together in a small dictionary and identify any contradiction with those counts. These synthetic results support no patient diagnosis or claim of clinical validity.

## 1. Understand the task

Read the input values below before running the request. This notebook includes all required setup and can be run independently. Companion files `interaction-traces-data.json` and `interaction-traces-reference.json` contain the same scenario and expected results.


**Downloaded package:** keep one private `.env` at the package root beside `COMPANION_VERSION`. The helper finds it automatically. When using this notebook alone, keep `.env` beside the notebook. The `get_completion()` interface is unchanged.

## Start here

This notebook includes its own setup. You do not need to run another notebook first. Start at the top and run each code cell in order, including the setup cells. To run everything, choose **Run → Run All Cells**. After restarting the kernel, run the setup again before running the experiment.

The default mode uses supplied examples and needs no API key. For OpenAI requests, follow the configuration instructions in this notebook and use a standard Python kernel.


## Provided setup — run once

Reuse `get_completion(system_prompt, user_prompt)` from Section 2.3. The shared notebook setup introduced in 3.2 adds supplied/live mode and records requests; its calling interface is unchanged here. Run the supplied helper cell, then the mode cell. You do not need to edit or retype the helper for live coding.

Default `"offline"` mode uses labelled supplied responses. For OpenAI, use a standard Python kernel, install `openai` and `python-dotenv`, and create `.env` beside the notebook containing `OPENAI_API_KEY=your_key` and `OPENAI_MODEL=your_model`. Set MODE to `"live"`. Browser Python supports the supplied examples. Never share a filled `.env` or print the key.

An edited request has no saved offline answer: the helper reports that no model was called. Do not evaluate that notice as a model response. Use live mode to test your changes.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
# Choose supplied examples or your own OpenAI account.
MODE = "offline"
if MODE == "live":
    configure_live()


In [ ]:
# Supplied classifier outcomes, not predictions from real patients.
counts = {"TP": 3, "FP": 1, "FN": 2, "TN": 4}
print("Supplied counts:", counts)


## 2. Predict

What would be wrong with a report claiming 100% accuracy and no missed positive cases? Identify the counts that settle this before inspecting the supplied trace.

## 3. Run

Use the same request function introduced earlier. Only the persona, data and the few method-specific lines below are new. The original supplied responses are teaching examples, not recorded live model results.


In [ ]:
system_prompt = "You explain supplied classification counts to students. Report accuracy, recall and missed positives. Do not invent counts, diagnose people or claim clinical validity."
user_prompt = f"Explain these fictional classification results: {counts}"


In [ ]:
# Supplied examples for these exact original requests; this does not call a model.
register_sample('You explain supplied classification counts to students. Report accuracy, recall and missed positives. Do not invent counts, diagnose people or claim clinical validity.', "Explain these fictional classification results: {'TP': 3, 'FP': 1, 'FN': 2, 'TN': 4}", 'The classifier has 100% accuracy and misses no positive cases.')


In [ ]:
# Calculate the reference metrics after making your prediction.
accuracy = (counts["TP"] + counts["TN"]) / sum(counts.values())
recall = counts["TP"] / (counts["TP"] + counts["FN"])
print("Expected accuracy:", accuracy)
print("Expected recall:", recall)
print("Missed positives:", counts["FN"])

# Keep the request and original response together; never record the API key.
response = get_completion(system_prompt, user_prompt)
trace = {"request": user_prompt, "response": response, "mode": MODE}
print("REQUEST\n", trace["request"])
print("RESPONSE\n", trace["response"])
print("MODE:", trace["mode"])


## 4. Check and explain

The reference is 70% accuracy, 60% recall and two missed positives. The supplied teaching response is deliberately wrong; a live response may be correct. The request contains the needed counts, so an unsupported report is not caused by missing input. A trace reveals the messages, not the model’s hidden reasoning or internal cause.

**Expected result:**

```json
{
  "accuracy": 0.7,
  "recall": 0.6,
  "missed_positives": 2,
  "clinical_validity": "Not established by this synthetic example."
}
```


## 5. Try one change

Add a requirement to show the arithmetic and retest the same counts live. State exactly which instruction changed. A correct report on ten synthetic cases does not establish clinical usefulness.

Submit your prediction, the observed result, one evidence-based check and a short explanation of your change. Label the response supplied or live, and mark any live test you did not perform. No additional project is required.
